# LC 424 — Longest Repeating Character Replacement

| Field      | Value                                         |
|------------|-----------------------------------------------|
| Difficulty | Medium                                        |
| Category   | String / Sliding Window                       |
| Pattern    | Monotonically non-decreasing window trick     |

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> A window of size W is valid if
<code>W - max_count <= k</code>, where max_count is the
frequency of the most common char in the window.
The window never needs to shrink below the best size found
so far — it only ever stays the same or grows.
</div>


## Official Problem Statement

You are given a string `s` and an integer `k`. You can choose
any character of the string and change it to any other uppercase
English character. You can perform this operation at most `k`
times.

Return the length of the longest substring containing the same
letter you can get after performing the above operations.

**Constraints:**
- `1 <= s.length <= 10^5`
- `s` consists of only uppercase English letters.
- `0 <= k <= s.length`


## What This Is Actually Asking

You want the longest contiguous run of a single letter,
but you are allowed to "fix" up to k characters in that run.
So the window can have minority characters — as long as there
are at most k of them, you can replace them all for free.
Maximize the window length under that budget.


## Walk Through an Example by Hand

`s = "AABABBA"`, `k = 1`

```
count = [0]*26, max_count=0, left=0, best=0

R=0 'A': count[A]=1  max_count=1  window=1  1-1=0<=1  best=1
R=1 'A': count[A]=2  max_count=2  window=2  2-2=0<=1  best=2
R=2 'B': count[B]=1  max_count=2  window=3  3-2=1<=1  best=3
R=3 'A': count[A]=3  max_count=3  window=4  4-3=1<=1  best=4
R=4 'B': count[B]=2  max_count=3  window=5  5-3=2 > 1  INVALID
         shrink: remove s[L=0]='A', count[A]=2, left=1
         window=4  4-3=1<=1  valid (window same size, not smaller)
R=5 'B': count[B]=3  max_count=3  window=5  5-3=2 > 1  INVALID
         shrink: remove s[L=1]='A', count[A]=1, left=2
         window=4  keep best=4
R=6 'A': count[A]=2  max_count=3  window=5  5-3=2 > 1  INVALID
         shrink: remove s[L=2]='B', count[B]=2, left=3
         window=4  keep best=4

Answer: 4  ("AABA" with one 'B' replaced, or "ABBA" with 'A')
```


## The Picture

Window condition: **window_size - max_count <= k**

```
s =  A  A  B  A  B  B  A
     0  1  2  3  4  5  6
k = 1

Valid window (minority chars <= k):
     [  A  A  B  A  ]          size=4, max='A'(3), replace 1 'B'
     L              R          4 - 3 = 1 <= k  ✓

One more step — invalid:
     [  A  A  B  A  B  ]       size=5, max='A'(3), need 2 replaces
     L                 R       5 - 3 = 2 > k  ✗

Shrink left by 1 (remove 'A'):
        [  A  B  A  B  ]       size=4, max='A'or'B'(2), 4-2=2 > k
        L              R       Still invalid! Shrink again.

Key trick: window size never goes BELOW best seen (4).
It shifts right as a fixed-size frame until it can grow again.
```

Formula to memorize:
```
replacements_needed = window_size - max_count
valid               = replacements_needed <= k
```


## When To Use This Pattern

- When you see **"longest substring after at most k changes"**,
  think **sliding window + max frequency tracking**.
- When validity is `window_size - dominant_count <= budget`,
  think **this exact pattern**.
- When shrinking the window below the current best is wasteful,
  think **monotonically non-decreasing window trick**.
- When the alphabet is small and fixed (26 letters),
  think **array of 26 ints instead of a dict**.


## The Approach

Keep a count array of 26 integers for char frequencies in the
window. Track `max_count` — the highest frequency seen.
If `window_size - max_count > k`, the window is invalid:
remove the leftmost character and advance `left` by one.
Crucially, never shrink the window below its current size —
just shift it right, so the answer is simply the final window
size (or the largest it ever reached).


In [ ]:
from typing import List          # standard type hints
from collections import Counter  # available if needed for clarity


In [ ]:
def test_harness(func):
    """Run a fixed suite of tests against func(s, k) -> int."""
    tests = [
        # (s, k, expected)
        ("ABAB",    2, 4),  # replace both B or both A
        ("AABABBA", 1, 4),  # classic
        ("AAAA",    0, 4),  # all same, no replacements needed
        ("ABCD",    0, 1),  # no replacements allowed
        ("ABCD",    3, 4),  # replace 3, all become same
        ("A",       0, 1),  # single char
        ("AABB",    1, 3),  # 'AAB' or 'ABB'
        ("AAABBBCCC", 2, 5), # replace 2 to extend run
    ]
    passed = 0
    for s, k, expected in tests:
        result = func(s, k)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"{status} | s={repr(s):<12} k={k} "
            f"expected={expected} got={result}"
        )
    print(f"\n{passed}/{len(tests)} tests passed.")


In [ ]:
def character_replacement(s: str, k: int) -> int:
    """
    Return the length of the longest substring of one repeated
    character achievable with at most k replacements.

    Approach:
        Sliding window. count[26] tracks char freqs in window.
        max_count = max char freq ever seen in any window.
        If window_size - max_count > k: shrink left by 1.
        Window never shrinks below best size (monotonic trick).

    Time:  O(n)   — single pass, each char processed once
    Space: O(26)  — fixed-size count array, effectively O(1)
    """
    pass


# --- debug runs (expected values in comments) ---
print(character_replacement("ABAB",    2))  # 4
print(character_replacement("AABABBA", 1))  # 4
print(character_replacement("AAAA",    0))  # 4
print(character_replacement("ABCD",    0))  # 1
print(character_replacement("ABCD",    3))  # 4


In [ ]:
# Uncomment and run when solution is ready
# test_harness(character_replacement)


## Complexity

| Approach                    | Time    | Space  |
|-----------------------------|---------|--------|
| Brute force (all substrings)| O(n^2)  | O(26)  |
| Sliding window (this)       | O(n)    | O(26)  |


## Real World Connection

In Citi's real-time telemetry platform, anomaly detection
often works by asking: "what is the longest run of healthy
heartbeats from an endpoint, even if we allow up to k
anomalous readings to be dismissed as noise?"
This is exactly the character replacement problem — the
dominant state is 'healthy', the k budget is the noise
tolerance, and the window length is the confidence interval.
Over 6,000 endpoints streaming via Kinesis, this O(n)
sliding window runs in a single pass per shard, making
it viable for real-time rolling SLA dashboards in AWS.
The monotonic window trick means the operator never needs
to rewind the stream — it only moves forward.


> **Simplicity and clarity is Gold.** — Sean's Study Mantra
